# 3モデルアンサンブル
yolo26m + RT-DETR + RF-DETR → WBF → 提出

In [1]:
import os
os.chdir(r'C:\compe')

import json
import numpy as np
import pandas as pd
import torch
import torchvision.transforms.functional as TF
from PIL import Image as PILImage
from ultralytics import YOLO, RTDETR
from rfdetr import RFDETRBase
from ensemble_boxes import weighted_boxes_fusion
from pathlib import Path
from tqdm import tqdm

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし')

with open(r'C:\compe\test_dataset.json') as f:
    test_data = json.load(f)

fname_to_id      = {img['file_name']: img['id'] for img in test_data['images']}
img_id_to_info   = {img['id']: img for img in test_data['images']}
yolo_to_category = sorted([c['id'] for c in test_data['categories']])

TEST_DIR   = r'C:\compe\images\test'
test_files = sorted(os.listdir(TEST_DIR))
print(f'テスト画像数: {len(test_files)}')

c:\Users\tamkn\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 4050 Laptop GPU
テスト画像数: 1425


In [2]:
# モデル読み込み
model_yolo = YOLO(r'C:\compe\runs\full2\weights\best.pt')
print('yolo26m loaded')

RTDETR_PATH = r'C:\compe\runs\rtdetr4\weights\best.pt'
USE_RTDETR = os.path.exists(RTDETR_PATH)
if USE_RTDETR:
    model_rtdetr = RTDETR(RTDETR_PATH)
    print('RT-DETR loaded')

model_rfdetr = RFDETRBase(
    pretrain_weights=r'C:\compe\runs\rfdetr\checkpoint_best_total.pth',
    num_classes=32
)
model_rfdetr.optimize_for_inference()
print('RF-DETR loaded')

yolo26m loaded
RT-DETR loaded


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`use_return_dict` is deprecated! Use `return_dict` instead!


RF-DETR loaded


In [3]:
# 推論設定
CONF             = 0.001
IOU              = 0.6
MAX_DET          = 1000
WBF_IOU          = 0.55
WBF_SKIP_BOX_THR = 0.005
RFDETR_THR       = 0.1
WEIGHTS          = [1.0, 0.5, 3.0]  # yolo26m : RT-DETR : RF-DETR

TTA_PATTERNS = [
    (False, 1024),
    (True,  1024),
    (False,  896),
]

In [4]:
# 推論関数
def predict_yolo_tta(model, img_path, conf, iou):
    img_orig = PILImage.open(img_path).convert('RGB')
    all_boxes, all_scores, all_labels = [], [], []

    for do_flip, sz in TTA_PATTERNS:
        img = img_orig.copy()
        if do_flip:
            img = TF.hflip(img)

        result = model.predict(
            source=img,
            conf=conf,
            iou=iou,
            max_det=MAX_DET,
            imgsz=sz,
            verbose=False,
            half=True,
        )[0]

        if len(result.boxes) == 0:
            continue

        boxes  = result.boxes.xyxyn.cpu().numpy().tolist()
        scores = result.boxes.conf.cpu().numpy().tolist()
        labels = result.boxes.cls.cpu().numpy().astype(int).tolist()

        if do_flip:
            boxes = [[1-x2, y1, 1-x1, y2] for x1, y1, x2, y2 in boxes]

        boxes = [[min(max(v, 0.0), 1.0) for v in box] for box in boxes]
        all_boxes.append(boxes)
        all_scores.append(scores)
        all_labels.append(labels)

    if not all_boxes:
        return [], [], []

    boxes_f, scores_f, labels_f = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels,
        weights=[1.0] * len(all_boxes),
        iou_thr=WBF_IOU,
        skip_box_thr=WBF_SKIP_BOX_THR,
    )
    return boxes_f.tolist(), scores_f.tolist(), labels_f.tolist()


def predict_rfdetr(model, img_path, w, h, threshold):
    img = PILImage.open(img_path).convert('RGB')
    result = model.predict(img, threshold=threshold)

    if len(result) == 0:
        return [], [], []

    valid = result.class_id < 32
    if not valid.any():
        return [], [], []

    boxes  = result.xyxy[valid].astype(float)
    boxes[:, [0, 2]] /= w
    boxes[:, [1, 3]] /= h
    boxes  = np.clip(boxes, 0, 1).tolist()
    scores = result.confidence[valid].tolist()
    labels = result.class_id[valid].tolist()

    return boxes, scores, labels

In [5]:
# 推論 & アンサンブル
rows = []

for fname in tqdm(test_files):
    img_path = os.path.join(TEST_DIR, fname)
    image_id = fname_to_id.get(fname)
    if image_id is None:
        stem = Path(fname).stem
        for key in fname_to_id:
            if Path(key).stem == stem:
                image_id = fname_to_id[key]
                break
    if image_id is None:
        continue

    w = img_id_to_info[image_id]['width']
    h = img_id_to_info[image_id]['height']

    boxes_y,  scores_y,  labels_y  = predict_yolo_tta(model_yolo, img_path, CONF, IOU)
    boxes_rf, scores_rf, labels_rf = predict_rfdetr(model_rfdetr, img_path, w, h, RFDETR_THR)

    if USE_RTDETR:
        boxes_r, scores_r, labels_r = predict_yolo_tta(model_rtdetr, img_path, CONF, IOU)
    else:
        boxes_r, scores_r, labels_r = [], [], []

    boxes_list, scores_list, labels_list, weights_used = [], [], [], []

    if len(boxes_y) > 0:
        boxes_list.append(boxes_y);  scores_list.append(scores_y);  labels_list.append(labels_y);  weights_used.append(WEIGHTS[0])
    if len(boxes_r) > 0 and USE_RTDETR:
        boxes_list.append(boxes_r);  scores_list.append(scores_r);  labels_list.append(labels_r);  weights_used.append(WEIGHTS[1])
    if len(boxes_rf) > 0:
        boxes_list.append(boxes_rf); scores_list.append(scores_rf); labels_list.append(labels_rf); weights_used.append(WEIGHTS[2])

    if not boxes_list:
        continue

    boxes_f, scores_f, labels_f = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        weights=weights_used,
        iou_thr=WBF_IOU,
        skip_box_thr=WBF_SKIP_BOX_THR,
    )

    for box, score, label in zip(boxes_f, scores_f, labels_f):
        x1, y1, x2, y2 = box
        rows.append({
            'image_id':    image_id,
            'category_id': yolo_to_category[int(label)],
            'bbox_x':      x1 * w,
            'bbox_y':      y1 * h,
            'bbox_width':  (x2 - x1) * w,
            'bbox_height': (y2 - y1) * h,
            'score':       float(score),
        })

print(f'予測数: {len(rows)}')
print(f'1画像あたり平均: {len(rows)/len(test_files):.1f}box')

100%|██████████| 1425/1425 [09:29<00:00,  2.50it/s]

予測数: 239321
1画像あたり平均: 167.9box


In [6]:
# 提出ファイル作成
submission = pd.DataFrame(rows)
submission['annotation_id'] = np.arange(len(submission))
submission = submission[[
    'annotation_id', 'image_id', 'category_id',
    'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'score'
]]
submission['score'] = submission['score'].clip(0, 1)
submission.to_csv(r'C:\compe\submission.csv', index=False)
print(f'完了！{len(submission)}行')
print(submission.head())

完了！239321行
   annotation_id  image_id  category_id       bbox_x      bbox_y  bbox_width  \
0              0         1           26   233.902516  614.178636  112.290530   
1              1         1            2   221.193552   42.845171   68.264294   
2              2         1           26   384.484205  408.903644  284.137659   
3              3         1           26   383.607666  392.280731   91.723663   
4              4         1            2  1524.595581   77.991257   55.432861   

   bbox_height     score  
0    84.044251  0.234631  
1    53.261900  0.179103  
2   190.054314  0.134500  
3    77.578033  0.108097  
4    51.127609  0.106934  
